In [ ]:
import sqlite3

conn = sqlite3.connect('news.sqlite3')
cursor = conn.cursor()
query = """
DELETE FROM articles 
WHERE id NOT IN (
    SELECT MIN(id) 
    FROM articles 
    GROUP BY heading
    )
"""
cursor.execute(query)
conn.commit()
query = """
DELETE FROM articles 
WHERE id NOT IN (
    SELECT MIN(id) 
    FROM articles 
    GROUP BY body
    )
"""
cursor.execute(query)
conn.commit()
query = """
DELETE FROM articles 
WHERE id NOT IN (
    SELECT MIN(id) 
    FROM articles 
    GROUP BY url
    )
"""
cursor.execute(query)
conn.commit()

query = """
SELECT * FROM articles
"""
cursor.execute(query)
rows = cursor.fetchall()

In [ ]:
import os
from api_keys import GEMINI_1
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage

os.environ["GOOGLE_API_KEY"] = GEMINI_1

llm = ChatGoogleGenerativeAI(
    model="models/gemini-3.1-flash-lite-preview",
    temperature=0.0,
    max_retries=6,
)

def ask_llm(question):
    try:
        message = HumanMessage(content=question)
        response = llm.invoke([message])
        return response.content
    except Exception as e:
        return f"Error: {e}"

In [ ]:
resp = []

In [ ]:
import math
it = math.ceil(len(rows) / 10)
i = 0

In [ ]:
for x in range(it):
    p = ""
    for r in rows[10*x:10*(x+1)]:
        p += f"--{r[3]}--\n{r[4]}\n===" 
    pytanie = f"""
        Jesteś osobą analizującą, jak są pisane artykuły w polskich mediach.
        Przeanalizuj podane artykuły, które zostaną podane poniżej. Podaj informację
        o czym jest artykuł (dokładnie jakiego wydarzenia w historii to dotyczy),
        w jakim stylu został napisany oraz jak długi on jest. Podaj odpowiedzi w formie,
        zdarzenie, temat, długość. Niech odpowiedzi będą budowane jak ta odpowiedź:
        Oto analiza podanych artykułów:\n\n1. **Zdarzenie:** Informacje praktyczne o godzinach otwarcia sklepów w Sylwestra i Nowy Rok.\n   **Temat:** Handel w okresie świąteczno-noworocznym.\n   **Długość:** Krótki artykuł informacyjny.\n\n2. **Zdarzenie:** Reakcja rosyjskiego MSZ na wypowiedzi ambasadorki USA w Polsce dotyczące II wojny światowej.\n   **Temat:** Spór dyplomatyczny i historyczny między Polską, USA a Rosją.\n   **Długość:** Długi artykuł publicystyczny.\n\n
        
        Artykuły do analizy:
    {p}"""

    t = ask_llm(pytanie)
    u = []
    for r in t[0]['text'][33:].split('\n\n'):
        u = [rows[i][0], rows[i][1], rows[i][2]] + [r]
        resp.append(u)
        i += 1
    print(f"{x} / {it}")

In [ ]:
import json
with open('articles_topics.json', 'w', encoding='utf-8') as f:
    json.dump(resp, f, ensure_ascii=False)

In [ ]:
to_save = []

for k in resp:
    p = {}
    p['ID'] = k[0]
    p['Provider'] = k[1]
    p['URL'] = k[2]
    splits = k[3].split('\n')
    p['Zdarzenie'] = splits[0].split('** ')[-1]
    p['Temat'] = splits[1].split('** ')[-1]
    p['Długość'] = splits[2].split('** ')[-1]
    to_save.append(p)

In [ ]:
import json
with open('articles_topics_processed.json', 'w', encoding='utf-8') as f:
    json.dump(to_save, f, ensure_ascii=False)